# Decision Consistency — Initial Prototype (CIFAR-10, ResNet-18)

Original prototype. Single model, single dataset, 3 perturbations, N=200.
Establishes the core pipeline: load, perturb, infer, compute metrics, Grad-CAM.
Baseline for all subsequent multi-model experiments.

**Update DATA_DIR before running.**

In [ ]:
DATA_DIR   = r'E:\cifar-10-python\cifar-10-batches-py'
OUTPUT_DIR = 'decision_consistency_outputs'
NUM_SAMPLES = 200
SEED = 42

In [ ]:
import os, pickle, json, random
import numpy as np
import torch, torch.nn.functional as F
import torchvision.transforms.functional as TF
from torchvision import models, transforms
from PIL import Image
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from tqdm import tqdm
import warnings; warnings.filterwarnings('ignore')
for sub in ['figures/confidence_plots','figures/gradcam_maps','tables']:
    os.makedirs(os.path.join(OUTPUT_DIR,sub),exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:',device)

In [ ]:
with open(os.path.join(DATA_DIR,'test_batch'),'rb') as f:
    batch=pickle.load(f,encoding='bytes')
X_test=batch[b'data'].reshape(-1,3,32,32); y_test=np.array(batch[b'labels'])
X_torch=torch.tensor(X_test,dtype=torch.float32)/255.0
print('Test set:',X_test.shape)

In [ ]:
model=models.resnet18(pretrained=True).eval().to(device)
preprocess=transforms.Compose([transforms.Resize((224,224)),transforms.ToTensor(),transforms.Normalize(mean=[0.485,0.456,0.406],std=[0.229,0.224,0.225])])
def prepare(t): return preprocess(transforms.ToPILImage()(t)).unsqueeze(0).to(device)
def infer(t):
    with torch.no_grad(): probs=F.softmax(model(prepare(t)),dim=1); return probs.argmax(1).item(),probs.max().item()
print('ResNet-18 loaded')

In [ ]:
# Original 3 perturbations from the initial paper
def generate_perturbations(t):
    noise=0.01*torch.randn_like(t)
    return {'original':t,'rotated':TF.rotate(t,angle=2),'bright':TF.adjust_brightness(t,brightness_factor=1.1),'noisy':torch.clamp(t+noise,0,1)}
def compute_metrics(results_dict):
    base_class=results_dict['original']['pred']; base_conf=results_dict['original']['conf']
    variants={k:v for k,v in results_dict.items() if k!='original'}; K=len(variants)
    same=sum(1 for v in variants.values() if v['pred']==base_class); diffs=[abs(v['conf']-base_conf) for v in variants.values()]
    S=same/K; D=sum(diffs)/K; C=S*(1-D)
    return {'Label_Stability':S,'Confidence_Deviation':D,'Consistency_Score':C}

In [ ]:
all_metrics=[]
for idx in tqdm(range(NUM_SAMPLES),desc='Evaluating'):
    img=X_torch[idx]; perts=generate_perturbations(img)
    results={k:{'pred':infer(v)[0],'conf':infer(v)[1]} for k,v in perts.items()}
    m=compute_metrics(results); m['true_label']=int(y_test[idx]); all_metrics.append(m)
ls_vals=[m['Label_Stability'] for m in all_metrics]; cd_vals=[m['Confidence_Deviation'] for m in all_metrics]; cs_vals=[m['Consistency_Score'] for m in all_metrics]
print(f'ResNet-18/CIFAR-10: LS={np.mean(ls_vals):.3f}  CD={np.mean(cd_vals):.3f}  CS={np.mean(cs_vals):.3f}+/-{np.std(cs_vals):.3f}')
with open(os.path.join(OUTPUT_DIR,'tables','raw_metrics.json'),'w') as f: json.dump(all_metrics,f,indent=2)

In [ ]:
for vals,fname,xlabel,title in [(cs_vals,'consistency_hist.png','Consistency Score','Distribution of CS'),(cd_vals,'confidence_dev_hist.png','Confidence Deviation','Distribution of CD')]:
    fig,ax=plt.subplots(figsize=(6,4)); ax.hist(vals,bins=20,color='steelblue',edgecolor='white'); ax.axvline(np.mean(vals),color='red',linestyle='--',lw=1.5,label=f'Mean={np.mean(vals):.3f}')
    ax.set_xlabel(xlabel); ax.set_ylabel('Frequency'); ax.set_title(title); ax.legend(); ax.grid(alpha=0.3); plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR,'figures','confidence_plots',fname),dpi=300,bbox_inches='tight'); plt.close(); print('Saved:',fname)

In [ ]:
class GradCAM:
    def __init__(self,model):
        self.model=model; self.grads=self.acts=None
        model.layer4[-1].register_forward_hook(lambda m,i,o: setattr(self,'acts',o.detach()))
        model.layer4[-1].register_backward_hook(lambda m,gi,go: setattr(self,'grads',go[0].detach()))
    def generate(self,inp,class_idx):
        self.model.zero_grad(); self.model(inp)[0,class_idx].backward()
        w=self.grads.mean(dim=[2,3],keepdim=True); cam=torch.relu((w*self.acts).sum(dim=1).squeeze()).cpu().numpy()
        cam=(cam-cam.min())/(cam.max()-cam.min()+1e-8)
        return np.array(Image.fromarray((cam*255).astype(np.uint8)).resize((224,224),Image.BILINEAR))/255.0
gcam=GradCAM(model)
bidx=next((i for i,m in enumerate(all_metrics) if m['Label_Stability']<1.0),0)
img_o=X_torch[bidx]; img_p=generate_perturbations(img_o)['rotated']
inp_o=prepare(img_o).requires_grad_(True); inp_p=prepare(img_p).requires_grad_(True)
po,_=infer(img_o); pp,_=infer(img_p); cam_o=gcam.generate(inp_o,po); cam_p=gcam.generate(inp_p,pp)
def denorm(t): arr=t.squeeze().permute(1,2,0).detach().cpu().numpy(); return np.clip(arr*np.array([0.229,0.224,0.225])+np.array([0.485,0.456,0.406]),0,1)
fig,axes=plt.subplots(1,2,figsize=(6,3))
for ax,inp_t,cam,title in [(axes[0],inp_o,cam_o,f'Original pred:{po}'),(axes[1],inp_p,cam_p,f'Rotation pred:{pp}')]:
    ax.imshow(denorm(inp_t)); ax.imshow(cam,cmap='jet',alpha=0.45); ax.set_title(title,fontsize=9); ax.axis('off')
fig.suptitle(f'Grad-CAM ResNet-18/CIFAR-10 sample#{bidx}',fontsize=10); plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'figures','gradcam_maps','gradcam.png'),dpi=300,bbox_inches='tight'); plt.close(); print('Grad-CAM saved.')